# Phase 1 optimizer -- cross-check ledger against the test suite

This is one of 7 notebooks in `notebooks/phase1_optimization/`.
Equation/section citations here follow `docs/tt pacing optimizer.md`, the
project's canonical numbering, written as "(design-doc Eq. N)" /
"(design-doc Section N.M)"; display-unit conventions (km/min/km-h⁻¹) are
implemented in `phase_1_0_common.py`'s `as_km`/`as_min`/`as_kmh` helpers, shared
by all 7 notebooks. This one covers
**the full numeric scoreboard: every claim made across the other 6 notebooks, cross-referenced against the pytest gate (if any) that backs it**.

It is fully self-contained: every rider/course/baseline it needs is
rebuilt here via `phase_1_0_common.py` (shared by all 7 split notebooks),
so it runs standalone from a fresh kernel without any other notebook
having run first. Courses used: all 5: flat, rolling (synthetic) + Giro10, TdF16, TARA (real). Approx. runtime: ~25-30 min -- as expensive as recomputing everything from scratch, since there is nothing to load from the other 6 notebooks (no cross-notebook caching by design).

This is the one notebook meant to be run rarely (a full sanity pass, e.g.
before a release), not for day-to-day iteration -- for that, use whichever
of the other 6 notebooks covers the thing you're actually debugging.

In [1]:
import sys
sys.path.insert(0, ".")
import phase_1_0_common as pc

pc.print_rider_summary()

from ttt_strat.collocation import CollocationProblem
from ttt_strat.optimizer import SLSQPSolver, _equilibrium_speed_and_relax_length, _graded_mesh
from ttt_strat.simulator import ForwardSimulator, _launch_and_truncate
import numpy as np
import matplotlib.pyplot as plt

reference_rider        mass= 72.0 kg  CP= 280.0 W  W'= 20.0 kJ  CdA=0.25 m^2  P_max=   900 W
evenepoel_like_rider   mass= 63.5 kg  CP= 425.0 W  W'= 20.0 kJ  CdA=0.21 m^2  P_max=  1400 W
ganna_like_rider       mass= 82.0 kg  CP= 480.0 W  W'= 20.0 kJ  CdA=0.19 m^2  P_max=  1600 W


In [2]:
courses = pc.load_real_courses()
giro10_course, tdf16_course, tara_course = courses["giro10"], courses["tdf16"], courses["tara"]
pc.print_real_course_summary(courses)

Giro 2026 Stage 10   n_nodes= 800  smoothing_length_m=  150 m  mean|grade|= 1.04%  max|grade|=13.20%
TdF 2026 Stage 16    n_nodes= 800  smoothing_length_m=  400 m  mean|grade|= 5.30%  max|grade|=18.53%
TARA 2026 Stage 3    n_nodes= 800  smoothing_length_m=  250 m  mean|grade|= 3.05%  max|grade|=17.56%


## Stage A -- scheme comparison (all 5 courses)

In [3]:
flat_course = pc.build_flat_course()
rolling_course = pc.build_rolling_course()
calm_wind = pc.calm_wind

res_hs_flat, res_trap_flat = pc.run_scheme_comparison(pc.reference_rider, flat_course, calm_wind, pc.N_INTERVALS_BASIC)
opt_hs_flat = pc.ITTOptimizer(pc.reference_rider, flat_course, calm_wind, scheme="hermite_simpson", solver="slsqp")
rel_diff_scheme_flat = pc.report_scheme_comparison(res_hs_flat, res_trap_flat, "flat")

res_hs_rolling, res_trap_rolling = pc.run_scheme_comparison(pc.reference_rider, rolling_course, calm_wind, pc.N_INTERVALS_BASIC)
opt_hs_rolling = pc.ITTOptimizer(pc.reference_rider, rolling_course, calm_wind, scheme="hermite_simpson", solver="slsqp")
rel_diff_scheme_rolling = pc.report_scheme_comparison(res_hs_rolling, res_trap_rolling, "rolling")

res_hs_giro10, res_trap_giro10 = pc.run_scheme_comparison(pc.ganna_like_rider, giro10_course, calm_wind, pc.N_INTERVALS_GIRO10)
opt_hs_giro10 = pc.ITTOptimizer(pc.ganna_like_rider, giro10_course, calm_wind, scheme="hermite_simpson", solver="slsqp")
rel_diff_scheme_giro10 = pc.report_scheme_comparison(res_hs_giro10, res_trap_giro10, "giro10")

res_hs_tdf16, res_trap_tdf16 = pc.run_scheme_comparison(pc.evenepoel_like_rider, tdf16_course, calm_wind, pc.N_INTERVALS_TDF16)
opt_hs_tdf16 = pc.ITTOptimizer(pc.evenepoel_like_rider, tdf16_course, calm_wind, scheme="hermite_simpson", solver="slsqp")
rel_diff_scheme_tdf16 = pc.report_scheme_comparison(res_hs_tdf16, res_trap_tdf16, "tdf16")

res_hs_tara, res_trap_tara = pc.run_scheme_comparison(pc.reference_rider, tara_course, calm_wind, pc.N_INTERVALS_TARA)
opt_hs_tara = pc.ITTOptimizer(pc.reference_rider, tara_course, calm_wind, scheme="hermite_simpson", solver="slsqp")
rel_diff_scheme_tara = pc.report_scheme_comparison(res_hs_tara, res_trap_tara, "tara")

[flat] Hermite-Simpson: T = 57.118 min (success=False)
[flat] Trapezoidal:     T = 57.080 min (success=False)
[flat] Relative difference: 0.00066  (test_trapezoidal_fallback_converges_near_hs_result gate: < 0.02)


[rolling] Hermite-Simpson: T = 34.056 min (success=False)
[rolling] Trapezoidal:     T = 34.027 min (success=True)
[rolling] Relative difference: 0.00084  (test_trapezoidal_fallback_converges_near_hs_result gate: < 0.02)


[giro10] Hermite-Simpson: T = 42.551 min (success=False)
[giro10] Trapezoidal:     T = 42.366 min (success=False)
[giro10] Relative difference: 0.00434  (test_trapezoidal_fallback_converges_near_hs_result gate: < 0.02)


[tdf16] Hermite-Simpson: T = 37.192 min (success=False)
[tdf16] Trapezoidal:     T = 37.215 min (success=True)
[tdf16] Relative difference: 0.00063  (test_trapezoidal_fallback_converges_near_hs_result gate: < 0.02)


[tara] Hermite-Simpson: T = 45.484 min (success=False)
[tara] Trapezoidal:     T = 45.610 min (success=True)
[tara] Relative difference: 0.00276  (test_trapezoidal_fallback_converges_near_hs_result gate: < 0.02)


## Stage B -- backend cross-validation (all 5 courses, reusing Stage A's `res_hs_*`)

In [4]:
rel_diff_backend_flat, rel_diff_sim_flat = pc.run_backend_and_sim_crosscheck(
    pc.reference_rider, flat_course, calm_wind, res_hs_flat, pc.N_INTERVALS_BASIC, "flat"
)
rel_diff_backend_rolling, rel_diff_sim_rolling = pc.run_backend_and_sim_crosscheck(
    pc.reference_rider, rolling_course, calm_wind, res_hs_rolling, pc.N_INTERVALS_BASIC, "rolling"
)
rel_diff_backend_giro10, rel_diff_sim_giro10 = pc.run_backend_and_sim_crosscheck(
    pc.ganna_like_rider, giro10_course, calm_wind, res_hs_giro10, pc.N_INTERVALS_GIRO10, "giro10"
)
rel_diff_backend_tdf16, rel_diff_sim_tdf16 = pc.run_backend_and_sim_crosscheck(
    pc.evenepoel_like_rider, tdf16_course, calm_wind, res_hs_tdf16, pc.N_INTERVALS_TDF16, "tdf16"
)
rel_diff_backend_tara, rel_diff_sim_tara = pc.run_backend_and_sim_crosscheck(
    pc.reference_rider, tara_course, calm_wind, res_hs_tara, pc.N_INTERVALS_TARA, "tara"
)


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit http://projects.coin-or.org/Ipopt
******************************************************************************



[flat] SLSQP: T = 57.1178 min  (success=False)
[flat] IPOPT: T = 57.1193 min  (success=False)
[flat] Relative difference: 0.000027  (test_ipopt_agrees_with_slsqp_within_tolerance gate: < 1e-3)
[flat] NLP (SLSQP) time:      57.1178 min
[flat] ForwardSimulator time: 57.1671 min
[flat] Relative difference: 0.000863  (test_slsqp_cross_validates_against_forward_simulator gate: < 0.005)


[rolling] SLSQP: T = 34.0558 min  (success=False)
[rolling] IPOPT: T = 34.0450 min  (success=False)
[rolling] Relative difference: 0.000317  (test_ipopt_agrees_with_slsqp_within_tolerance gate: < 1e-3)
[rolling] NLP (SLSQP) time:      34.0558 min
[rolling] ForwardSimulator time: 34.1290 min
[rolling] Relative difference: 0.002150  (test_slsqp_cross_validates_against_forward_simulator gate: < 0.005)


[giro10] SLSQP: T = 42.5510 min  (success=False)
[giro10] IPOPT: T = 42.9120 min  (success=False)
[giro10] Relative difference: 0.008486  (test_ipopt_agrees_with_slsqp_within_tolerance gate: < 1e-3)
[giro10] NLP (SLSQP) time:      42.5510 min
[giro10] ForwardSimulator time: 42.5155 min
[giro10] Relative difference: 0.000835  (test_slsqp_cross_validates_against_forward_simulator gate: < 0.005)


[tdf16] SLSQP: T = 37.1917 min  (success=False)
[tdf16] IPOPT: T = 37.1476 min  (success=False)
[tdf16] Relative difference: 0.001184  (test_ipopt_agrees_with_slsqp_within_tolerance gate: < 1e-3)
[tdf16] NLP (SLSQP) time:      37.1917 min
[tdf16] ForwardSimulator time: 37.2260 min
[tdf16] Relative difference: 0.000924  (test_slsqp_cross_validates_against_forward_simulator gate: < 0.005)


[tara] SLSQP: T = 45.4841 min  (success=False)
[tara] IPOPT: T = 42.7715 min  (success=False)
[tara] Relative difference: 0.059639  (test_ipopt_agrees_with_slsqp_within_tolerance gate: < 1e-3)
[tara] NLP (SLSQP) time:      45.4841 min
[tara] ForwardSimulator time: 44.2645 min
[tara] Relative difference: 0.026813  (test_slsqp_cross_validates_against_forward_simulator gate: < 0.005)


## Stage C -- launch mesh grading

Builds Giro10/TdF16 with the mesh-grading/real-course-validation
notebooks' shared `smoothing_length_m=300` processing (distinct from
Stage A's `giro10_course`/`tdf16_course`, which use the lighter/heavier
150 m/400 m choice) -- reused again in Stage F below.

In [5]:
giro10_data_300 = pc.load_gpx(pc.GPX_DIR / "giro2026_stage10.gpx")
giro10_course_300 = pc.CourseProcessor().process(
    giro10_data_300, n_nodes=len(giro10_data_300.s_m), smoothing_length_m=300.0
)
tdf16_data_300 = pc.load_gpx(pc.GPX_DIR / "tdf2026_stage16.gpx")
tdf16_course_300 = pc.CourseProcessor().process(
    tdf16_data_300, n_nodes=min(len(tdf16_data_300.s_m), 800), smoothing_length_m=300.0
)

opt_giro10 = pc.ITTOptimizer(pc.ganna_like_rider, giro10_course_300, calm_wind, scheme="hermite_simpson", solver="slsqp")
res_giro10_baseline = opt_giro10.optimize(n_intervals=80)

opt_tdf16 = pc.ITTOptimizer(pc.evenepoel_like_rider, tdf16_course_300, calm_wind, scheme="hermite_simpson", solver="slsqp")
res_tdf16_baseline = opt_tdf16.optimize(n_intervals=60)

v_w_flat = calm_wind.head_wind_m_per_s(flat_course.bearing_rad)
t_match_s, s_match_m, w_after_launch_J, i_start = _launch_and_truncate(
    pc.reference_rider, flat_course, v_w_flat, 1.225, opt_hs_flat.v_match_m_per_s
)
s_sub = flat_course.s_m[i_start:]
theta_sub = flat_course.theta_rad[i_start:]
vw_sub = v_w_flat[i_start:]

v_eq_m_per_s, l_relax_m = _equilibrium_speed_and_relax_length(
    pc.reference_rider, float(theta_sub[0]), float(vw_sub[0]), 1.225, pc.reference_rider.cp_W
)


def solve_on_mesh(s_new):
    v_max_m_per_s = max(v_eq_m_per_s * 2.5, 15.0)
    theta_new = np.interp(s_new, s_sub, theta_sub)
    vw_new = np.interp(s_new, s_sub, vw_sub)
    problem = CollocationProblem(
        pc.reference_rider, s_new, theta_new, vw_new, 1.225,
        v0_m_per_s=opt_hs_flat.v_match_m_per_s, w0_J=w_after_launch_J,
        scheme="hermite_simpson", v_max_m_per_s=v_max_m_per_s,
    )
    v_guess, w_guess, p_guess = opt_hs_flat._warm_start(
        s_new, theta_new, vw_new, opt_hs_flat.v_match_m_per_s, w_after_launch_J
    )
    p_mid_guess = 0.5 * (p_guess[:-1] + p_guess[1:])
    z0 = problem.pack(v_guess, w_guess, p_guess, p_mid_guess)
    z_opt, success, _message = SLSQPSolver().solve(problem, z0)
    _v_opt, _w_opt, p_opt, p_mid_opt = problem.unpack(z_opt)
    t_nlp_s = t_match_s + problem.objective(z_opt)

    full_power_W = np.empty(len(flat_course.s_m))
    full_power_W[:i_start] = p_opt[0]
    s_ctrl = np.empty(2 * problem.n_intervals + 1)
    p_ctrl = np.empty(2 * problem.n_intervals + 1)
    s_ctrl[0::2], p_ctrl[0::2] = s_new, p_opt
    s_ctrl[1::2] = 0.5 * (s_new[:-1] + s_new[1:])
    p_ctrl[1::2] = p_mid_opt
    full_power_W[i_start:] = np.interp(flat_course.s_m[i_start:], s_ctrl, p_ctrl)

    sim = ForwardSimulator().simulate(pc.reference_rider, flat_course, calm_wind, full_power_W)
    return p_opt, t_nlp_s, sim.time_total_s, success


s_graded = _graded_mesh(s_sub[0], s_sub[-1], pc.N_INTERVALS_BASIC, l_relax_m)
s_uniform = np.linspace(s_sub[0], s_sub[-1], pc.N_INTERVALS_BASIC + 1)

_p_graded, t_nlp_graded, t_sim_graded, success_graded = solve_on_mesh(s_graded)
_p_uniform, t_nlp_uniform, t_sim_uniform, success_uniform = solve_on_mesh(s_uniform)

rel_graded = abs(t_sim_graded - t_nlp_graded) / t_nlp_graded
rel_uniform = abs(t_sim_uniform - t_nlp_uniform) / t_nlp_uniform
print(f"GRADED  mesh: rel_diff={rel_graded:.5f}  success={success_graded}")
print(f"UNIFORM mesh: rel_diff={rel_uniform:.5f}  success={success_uniform}")

GRADED  mesh: rel_diff=0.00086  success=False
UNIFORM mesh: rel_diff=0.02771  success=False


## Stage D -- constrained re-optimization tiers (all 5 courses)

In [6]:
constrained_results_flat = pc.run_constrained_tiers(opt_hs_flat, res_hs_flat, catastrophic_mode="square", label="flat")
constrained_results_rolling = pc.run_constrained_tiers(opt_hs_rolling, res_hs_rolling, catastrophic_mode="sine", label="rolling")
constrained_results_giro10 = pc.run_constrained_tiers(opt_hs_giro10, res_hs_giro10, catastrophic_mode="square", label="giro10")
constrained_results_tdf16 = pc.run_constrained_tiers(opt_hs_tdf16, res_hs_tdf16, catastrophic_mode="sine", label="tdf16")
constrained_results_tara = pc.run_constrained_tiers(opt_hs_tara, res_hs_tara, catastrophic_mode="sine", label="tara")

[flat] good         input |dP/ds|: p95=  0.040 max=    7.52 W/m  ->  output max |dP/ds|=2.0000 W/m (bound 2.0)   T=57.1244 min (unsmoothed 57.1178 min)   delta= +0.398 s


[flat] acceptable   input |dP/ds|: p95=  0.098 max=    7.49 W/m  ->  output max |dP/ds|=2.0000 W/m (bound 2.0)   T=57.1244 min (unsmoothed 57.1178 min)   delta= +0.397 s


[flat] borderline   input |dP/ds|: p95=  0.184 max=    6.89 W/m  ->  output max |dP/ds|=2.0000 W/m (bound 2.0)   T=57.1244 min (unsmoothed 57.1178 min)   delta= +0.398 s


[flat] rough        input |dP/ds|: p95=  0.436 max=    8.99 W/m  ->  output max |dP/ds|=2.0000 W/m (bound 2.0)   T=57.1244 min (unsmoothed 57.1178 min)   delta= +0.397 s


[flat] very_rough   input |dP/ds|: p95=  4.635 max=   10.76 W/m  ->  output max |dP/ds|=2.0000 W/m (bound 2.0)   T=57.1247 min (unsmoothed 57.1178 min)   delta= +0.412 s


...

[tara] acceptable   input |dP/ds|: p95=  0.614 max=  105.64 W/m  ->  output max |dP/ds|=2.0000 W/m (bound 2.0)   T=45.4926 min (unsmoothed 45.4841 min)   delta= +0.509 s


[tara] borderline   input |dP/ds|: p95=  0.700 max=  102.53 W/m  ->  output max |dP/ds|=2.0000 W/m (bound 2.0)   T=45.4926 min (unsmoothed 45.4841 min)   delta= +0.509 s


[tara] rough        input |dP/ds|: p95=  0.842 max=   95.55 W/m  ->  output max |dP/ds|=2.0000 W/m (bound 2.0)   T=45.4926 min (unsmoothed 45.4841 min)   delta= +0.509 s


[tara] very_rough   input |dP/ds|: p95=  2.210 max=   87.99 W/m  ->  output max |dP/ds|=2.0000 W/m (bound 2.0)   T=45.4926 min (unsmoothed 45.4841 min)   delta= +0.509 s


[tara] catastrophic input |dP/ds|: p95=  1.735 max=   86.88 W/m  ->  output max |dP/ds|=5.7080 W/m (bound 2.0)   T=41.6125 min (unsmoothed 45.4841 min)   delta=-232.298 s


## Stage E -- post-hoc smoothing tiers (all 5 courses)

In [7]:
posthoc_results_flat = pc.run_posthoc_tiers(pc.reference_rider, flat_course, calm_wind, res_hs_flat, catastrophic_mode="square", label="flat")
posthoc_results_rolling = pc.run_posthoc_tiers(pc.reference_rider, rolling_course, calm_wind, res_hs_rolling, catastrophic_mode="sine", label="rolling")
posthoc_results_giro10 = pc.run_posthoc_tiers(pc.ganna_like_rider, giro10_course, calm_wind, res_hs_giro10, catastrophic_mode="square", label="giro10")
posthoc_results_tdf16 = pc.run_posthoc_tiers(pc.evenepoel_like_rider, tdf16_course, calm_wind, res_hs_tdf16, catastrophic_mode="sine", label="tdf16")
posthoc_results_tara = pc.run_posthoc_tiers(pc.reference_rider, tara_course, calm_wind, res_hs_tara, catastrophic_mode="sine", label="tara")

[flat] good         T= 57.0714 min  unsmoothed= 57.1178 min  delta= -2.787 s   w_prime_violated=True   frac_of_course_at_W'=0: 0.595
[flat] acceptable   T= 57.0876 min  unsmoothed= 57.1178 min  delta= -1.814 s   w_prime_violated=True   frac_of_course_at_W'=0: 0.295
[flat] borderline   T= 57.1114 min  unsmoothed= 57.1178 min  delta= -0.387 s   w_prime_violated=True   frac_of_course_at_W'=0: 0.228
[flat] rough        T= 57.1150 min  unsmoothed= 57.1178 min  delta= -0.171 s   w_prime_violated=True   frac_of_course_at_W'=0: 0.215
[flat] very_rough   T= 57.0865 min  unsmoothed= 57.1178 min  delta= -1.880 s   w_prime_violated=True   frac_of_course_at_W'=0: 0.590
[flat] catastrophic T= 47.8591 min  unsmoothed= 57.1178 min  delta=-555.521 s   w_prime_violated=True   frac_of_course_at_W'=0: 0.955
[rolling] good         T= 34.0056 min  unsmoothed= 34.0558 min  delta= -3.009 s   w_prime_violated=True   frac_of_course_at_W'=0: 0.233
[rolling] acceptable   T= 34.0099 min  unsmoothed= 34.0558 min  d

## Stage F -- real-course validation vs actual race results

Reuses Stage C's `giro10_course_300`/`tdf16_course_300`/
`res_giro10_baseline`/`res_tdf16_baseline`; TARA is built fresh with its
own `smoothing_length_m=200`.

In [8]:
tara_data = pc.load_gpx(pc.GPX_DIR / "tara2026_stage3.gpx")
tara_course_200 = pc.CourseProcessor().process(
    tara_data, n_nodes=len(tara_data.s_m), smoothing_length_m=200.0
)
opt_tara = pc.ITTOptimizer(pc.reference_rider, tara_course_200, calm_wind, scheme="hermite_simpson", solver="slsqp")
res_tara = opt_tara.optimize(n_intervals=80)

real_courses = [
    ("TARA 2026 Stage 3 (TTT)", tara_course_200, res_tara, 1972.17, (1.0, 1.5)),
    ("TdF 2026 Stage 16 (ITT)", tdf16_course_300, res_tdf16_baseline, 1939.0, (0.82, 1.18)),
    ("Giro 2026 Stage 10 (ITT)", giro10_course_300, res_giro10_baseline, 2753.0, (0.85, 1.15)),
]
for label, _course, res, real_time_s, (lo, hi) in real_courses:
    ratio = res.time_total_s / real_time_s
    print(f"{label:28s} ratio={ratio:.3f}  band=({lo}, {hi})  in_band={lo < ratio < hi}")

TARA 2026 Stage 3 (TTT)      ratio=1.397  band=(1.0, 1.5)  in_band=True
TdF 2026 Stage 16 (ITT)      ratio=1.155  band=(0.82, 1.18)  in_band=True
Giro 2026 Stage 10 (ITT)     ratio=0.931  band=(0.85, 1.15)  in_band=True


## Cross-check ledger

In [ ]:
cross_check_rows = [
    ("Sec 1.2", "trapezoidal vs Hermite-Simpson (flat)", rel_diff_scheme_flat, "< 0.02",
     "test_trapezoidal_fallback_converges_near_hs_result"),
    ("Sec 1.3", "trapezoidal vs Hermite-Simpson (rolling)", rel_diff_scheme_rolling, "< 0.02",
     "test_trapezoidal_fallback_converges_near_hs_result"),
    ("Sec 2.1", "SLSQP vs IPOPT (flat)", rel_diff_backend_flat, "< 1e-3",
     "test_ipopt_agrees_with_slsqp_within_tolerance"),
    ("Sec 2.1", "NLP vs ForwardSimulator (flat)", rel_diff_sim_flat, "< 0.005",
     "test_slsqp_cross_validates_against_forward_simulator"),
    ("Sec 2.2", "SLSQP vs IPOPT (rolling)", rel_diff_backend_rolling, "< 1e-3",
     "test_ipopt_agrees_with_slsqp_within_tolerance"),
    ("Sec 2.2", "NLP vs ForwardSimulator (rolling)", rel_diff_sim_rolling, "< 0.005",
     "test_slsqp_cross_validates_against_forward_simulator"),
    ("Sec 1.4", "trapezoidal vs Hermite-Simpson (Giro10, real)", rel_diff_scheme_giro10, "< 0.02",
     "test_trapezoidal_fallback_converges_near_hs_result"),
    ("Sec 1.5", "trapezoidal vs Hermite-Simpson (TdF16, real)", rel_diff_scheme_tdf16, "< 0.02",
     "test_trapezoidal_fallback_converges_near_hs_result"),
    ("Sec 1.6", "trapezoidal vs Hermite-Simpson (TARA, real)", rel_diff_scheme_tara, "< 0.02",
     "test_trapezoidal_fallback_converges_near_hs_result"),
    ("Sec 2.3", "SLSQP vs IPOPT (Giro10, real)", rel_diff_backend_giro10, "< 1e-3",
     "test_ipopt_agrees_with_slsqp_within_tolerance"),
    ("Sec 2.3", "NLP vs ForwardSimulator (Giro10, real)", rel_diff_sim_giro10, "< 0.005",
     "test_slsqp_cross_validates_against_forward_simulator"),
    ("Sec 2.4", "SLSQP vs IPOPT (TdF16, real)", rel_diff_backend_tdf16, "< 1e-3",
     "test_ipopt_agrees_with_slsqp_within_tolerance"),
    ("Sec 2.4", "NLP vs ForwardSimulator (TdF16, real)", rel_diff_sim_tdf16, "< 0.005",
     "test_slsqp_cross_validates_against_forward_simulator"),
    ("Sec 2.5", "SLSQP vs IPOPT (TARA, real)", rel_diff_backend_tara, "< 1e-3",
     "test_ipopt_agrees_with_slsqp_within_tolerance"),
    ("Sec 2.5", "NLP vs ForwardSimulator (TARA, real)", rel_diff_sim_tara, "< 0.005",
     "test_slsqp_cross_validates_against_forward_simulator"),
    ("Sec 1.2", "power_W <= p_max_W (flat)", float(np.max(res_hs_flat.power_W - pc.reference_rider.p_max_W)), "<= 1e-6",
     "test_optimizer_uses_rider_p_max_w_as_control_bound"),
    ("Sec 1.2", "terminal W'_bal / w_prime_J (flat)", res_hs_flat.w_prime_bal_J[-1] / pc.reference_rider.w_prime_J, "< 0.05",
     "test_slsqp_terminal_w_prime_near_zero"),
    ("Sec 1.2", "launch burst power > CP (flat)", res_hs_flat.power_W[:9].max() - pc.reference_rider.cp_W, "> 0",
     "test_slsqp_launch_region_shows_expected_bang_bang_burst"),
    ("Sec 1.4", "power_W <= p_max_W (Giro10, real)", float(np.max(res_hs_giro10.power_W - pc.ganna_like_rider.p_max_W)), "<= 1e-6",
     "test_optimizer_uses_rider_p_max_w_as_control_bound"),
    ("Sec 1.4", "terminal W'_bal / w_prime_J (Giro10, real)", res_hs_giro10.w_prime_bal_J[-1] / pc.ganna_like_rider.w_prime_J, "< 0.05",
     "test_slsqp_terminal_w_prime_near_zero"),
    ("Sec 1.5", "power_W <= p_max_W (TdF16, real)", float(np.max(res_hs_tdf16.power_W - pc.evenepoel_like_rider.p_max_W)), "<= 1e-6",
     "test_optimizer_uses_rider_p_max_w_as_control_bound"),
    ("Sec 1.5", "terminal W'_bal / w_prime_J (TdF16, real)", res_hs_tdf16.w_prime_bal_J[-1] / pc.evenepoel_like_rider.w_prime_J, "< 0.05",
     "test_slsqp_terminal_w_prime_near_zero"),
    ("Sec 1.6", "power_W <= p_max_W (TARA, real)", float(np.max(res_hs_tara.power_W - pc.reference_rider.p_max_W)), "<= 1e-6",
     "test_optimizer_uses_rider_p_max_w_as_control_bound"),
    ("Sec 1.6", "terminal W'_bal / w_prime_J (TARA, real)", res_hs_tara.w_prime_bal_J[-1] / pc.reference_rider.w_prime_J, "< 0.05",
     "test_slsqp_terminal_w_prime_near_zero"),
    ("Sec 3", "NLP vs ForwardSimulator, GRADED launch mesh", rel_graded, "(no direct test; ~0.09-0.3% per commit)",
     "-- (illustration only, see note below)"),
    ("Sec 3", "NLP vs ForwardSimulator, UNIFORM launch mesh (failure case)", rel_uniform, "(no test; expected to be much larger)",
     "-- (illustration only, see note below)"),
    ("Sec 4.1", "constrained re-opt max |dP/ds| (flat)", max(pc.slew_stats(sm.power_W, sm.s_m)[1] for _p, sm in constrained_results_flat.values()), "<= 2.0 * 1.01",
     "test_smoothing_constrained_satisfies_slew_bound"),
    ("Sec 4.1", "constrained re-opt time_total_s >= unsmoothed, all tiers (flat)",
     float(all(sm.time_total_s >= sm.unsmoothed_time_total_s for _p, sm in constrained_results_flat.values())),
     "True", "test_smoothing_constrained_satisfies_slew_bound"),
    ("Sec 4.2", "constrained re-opt max |dP/ds| (rolling)", max(pc.slew_stats(sm.power_W, sm.s_m)[1] for _p, sm in constrained_results_rolling.values()), "<= 2.0 * 1.01",
     "test_smoothing_constrained_satisfies_slew_bound"),
    ("Sec 4.3", "constrained re-opt max |dP/ds| (Giro10, real)", max(pc.slew_stats(sm.power_W, sm.s_m)[1] for _p, sm in constrained_results_giro10.values()), "<= 2.0 * 1.01",
     "test_smoothing_constrained_satisfies_slew_bound"),
    ("Sec 4.4", "constrained re-opt max |dP/ds| (TdF16, real)", max(pc.slew_stats(sm.power_W, sm.s_m)[1] for _p, sm in constrained_results_tdf16.values()), "<= 2.0 * 1.01",
     "test_smoothing_constrained_satisfies_slew_bound"),
    ("Sec 4.5", "constrained re-opt max |dP/ds| (TARA, real)", max(pc.slew_stats(sm.power_W, sm.s_m)[1] for _p, sm in constrained_results_tara.values()), "<= 2.0 * 1.01",
     "test_smoothing_constrained_satisfies_slew_bound"),
    ("Sec 5.1", "post-hoc: slower OR w_prime_violated, all tiers (flat)",
     float(all(sm.time_total_s >= sm.unsmoothed_time_total_s or sm.w_prime_violated for _p, sm, _s in posthoc_results_flat.values())),
     "True", "test_smoothing_posthoc_slower_or_flagged_infeasible"),
    ("Sec 5.2", "post-hoc: slower OR w_prime_violated, all tiers (rolling)",
     float(all(sm.time_total_s >= sm.unsmoothed_time_total_s or sm.w_prime_violated for _p, sm, _s in posthoc_results_rolling.values())),
     "True", "test_smoothing_posthoc_slower_or_flagged_infeasible"),
    ("Sec 5.3", "post-hoc: slower OR w_prime_violated, all tiers (Giro10, real)",
     float(all(sm.time_total_s >= sm.unsmoothed_time_total_s or sm.w_prime_violated for _p, sm, _s in posthoc_results_giro10.values())),
     "True", "test_smoothing_posthoc_slower_or_flagged_infeasible"),
    ("Sec 5.4", "post-hoc: slower OR w_prime_violated, all tiers (TdF16, real)",
     float(all(sm.time_total_s >= sm.unsmoothed_time_total_s or sm.w_prime_violated for _p, sm, _s in posthoc_results_tdf16.values())),
     "True", "test_smoothing_posthoc_slower_or_flagged_infeasible"),
    ("Sec 5.5", "post-hoc: slower OR w_prime_violated, all tiers (TARA, real)",
     float(all(sm.time_total_s >= sm.unsmoothed_time_total_s or sm.w_prime_violated for _p, sm, _s in posthoc_results_tara.values())),
     "True", "test_smoothing_posthoc_slower_or_flagged_infeasible"),
    ("Sec 6", "TARA 2026 Stage 3 ratio vs real result", res_tara.time_total_s / 1972.17, "(1.0, 1.5)",
     "test_tara2026_stage3_solo_slower_than_drafting_team_ballpark"),
    ("Sec 6", "TdF 2026 Stage 16 ratio vs real result", res_tdf16_baseline.time_total_s / 1939.0, "(0.82, 1.18)",
     "test_tdf2026_stage16_ballpark_vs_real_result"),
    ("Sec 6", "Giro 2026 Stage 10 ratio vs real result", res_giro10_baseline.time_total_s / 2753.0, "(0.85, 1.15)",
     "test_giro2026_stage10_ballpark_vs_real_result"),
]

print(f"{'section':8s} {'claim':52s} {'value':>12s}  {'test gate':30s} {'test function'}")
print("-" * 135)
for section, claim, value, gate, test_fn in cross_check_rows:
    print(f"{section:8s} {claim:52s} {value:12.5f}  {gate:30s} {test_fn}")

**Not cross-checked against a test**: the launch-mesh failure case (the
`UNIFORM` mesh row above) has no corresponding assertion in
`tests/test_phase_1*.py` -- it was constructed directly from
`optimizer._graded_mesh`/`_equilibrium_speed_and_relax_length` specifically
to illustrate the solver pathology `_graded_mesh`'s own docstring
describes. The rolling-course tiers use the same test-derived tolerances
as their flat-course counterparts -- those tests assert relative
agreement/bounds that are not specific to any one course -- but the small
(<0.1%) negative time deltas discussed in the constrained-smoothing
notebook's 4.2 closing note are this notebook set's own observation, not
something any test asserts either way. The same applies to the
real-course tiers, and to their `catastrophic_mode` choice -- each
course's mode is picked empirically, the same way 4.2 picked `"sine"` for
the synthetic rolling course, not from a test.

**Genuine gate failures, reported as such, not hidden**:
`test_ipopt_agrees_with_slsqp_within_tolerance` is only ever asserted by
the real test suite against the synthetic flat-course fixture
(`tests/test_phase_1.py`); none of the three real courses actually meet
its <0.1% bound (Sec 2.3-2.5: 0.85%, 0.12%, 5.96%), and TARA also misses
the NLP-vs-simulator <0.5% bound (Sec 2.5: 2.68%). Sec 4.5's
constrained-re-optimization slew-bound row (5.71 W/m vs. the <=2.02 W/m
gate that Giro10, TdF16, and both synthetic courses all satisfy) is
likewise a real, reproducible failure on TARA, not a rounding artifact --
see the constrained-smoothing notebook's 4.4/4.5 closing notes for the
specific non-convergence pathologies behind these numbers. Every other
numeric claim in this notebook set is backed by the test listed next to
it above, using the same tolerance that test enforces.